In [1]:
import os
import json
import random
from collections import defaultdict, Counter

In [2]:
!git clone https://github.com/RUCAIBox/POPE.git
%cd POPE
!python main.py \
    --seg_path ./segmentation/coco_ground_truth_segmentation.json \
    --sample_num 3 --img_num 500 --dataset coco \
    --save_path /kaggle/working/pope_output/
%cd ..

fatal: destination path 'POPE' already exists and is not an empty directory.
/kaggle/working/POPE
/kaggle/working


In [3]:
COCO_BASE = "/kaggle/input/datasets/nadaibrahim/coco2014"

def find_file(base, filename):
    for root, _, files in os.walk(base):
        if filename in files:
            return os.path.join(root, filename)
    return None

ann_path = find_file(COCO_BASE, "instances_val2014.json")
img_dir = os.path.dirname(find_file(COCO_BASE, "COCO_val2014_000000000139.jpg") or "")

with open(ann_path) as f:
    coco = json.load(f)

cat_id_to_name = {c["id"]: c["name"] for c in coco["categories"]}

image_to_objects = defaultdict(set)
for ann in coco["annotations"]:
    image_to_objects[ann["image_id"]].add(cat_id_to_name[ann["category_id"]])

image_id_to_filename = {img["id"]: img["file_name"] for img in coco["images"]}
coco_categories = set(cat_id_to_name.values())

print(len(image_to_objects), "images with ground-truth objects,", len(coco_categories), "categories")

40137 images with ground-truth objects, 80 categories


In [4]:
!git clone https://github.com/nickjiang2378/vlm-hallucinations.git

import re

with open("vlm-hallucinations/metric/chair.py") as f:
    content = f.read()

match = re.search(r"synonyms_txt = '''(.*?)'''", content, re.DOTALL)
synonyms_txt = match.group(1)


def build_synonym_dict(synonyms_txt: str) -> dict:
    synonym_dict = {}
    for line in synonyms_txt.strip().split("\n"):
        words = [w.strip() for w in line.split(",") if w.strip()]
        if not words:
            continue
        canonical = words[0]
        for w in words:
            synonym_dict[w] = canonical
    return synonym_dict


synonym_dict = build_synonym_dict(synonyms_txt)
print(len(synonym_dict), "synonym entries")

fatal: destination path 'vlm-hallucinations' already exists and is not an empty directory.
403 synonym entries


In [5]:
import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [6]:
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()


def extract_coco_objects_direct(caption: str, synonym_dict: dict, coco_categories: set) -> set:
    tokens = word_tokenize(caption.lower())
    lemmatized = [lemmatizer.lemmatize(t) for t in tokens]
    mapped = set()
    for tok in lemmatized:
        if tok in coco_categories:
            mapped.add(tok)
        elif tok in synonym_dict:
            mapped.add(synonym_dict[tok])
    return mapped


def label_caption(image_id: int, caption: str, image_to_objects: dict, synonym_dict: dict, coco_categories: set) -> list:
    mentioned = extract_coco_objects_direct(caption, synonym_dict, coco_categories)
    ground_truth = image_to_objects.get(image_id, set())
    return [
        {"image_id": image_id, "object": obj, "real": obj in ground_truth}
        for obj in mentioned
    ]


test_caption = "A dog sits next to a wooden bench in a park. A frisbee lies nearby."
sample_id = coco["images"][0]["id"]
for row in label_caption(sample_id, test_caption, image_to_objects, synonym_dict, coco_categories):
    print(row)

{'image_id': 391895, 'object': 'bench', 'real': False}
{'image_id': 391895, 'object': 'dog', 'real': False}
{'image_id': 391895, 'object': 'frisbee', 'real': False}


In [7]:
def label_pope_questions(path: str) -> list:
    """POPE's own yes/no questions, in the unified question-list schema. This is the
    PRIMARY object-hallucination baseline -- discriminative, same reasoning as why AMBER's
    and Reefknot's discriminative tasks are primary for attribute/relation."""
    rows = []
    with open(path) as f:
        for i, line in enumerate(f):
            d = json.loads(line)
            rows.append({
                "image_id": os.path.splitext(d["image"])[0],
                "question_id": f"pope_{i}",
                "question_text": d["text"],
                "ground_truth_answer": d["label"],  # "yes" / "no"
                "answer_format": "yes_no",
                "hallucination_type": "object",
            })
    return rows


pope_questions = label_pope_questions("/kaggle/working/pope_output/coco/coco_pope_random.json")
print(len(pope_questions), "POPE questions loaded")
print(pope_questions[0])

3000 POPE questions loaded
{'image_id': 'COCO_val2014_000000306812', 'question_id': 'pope_0', 'question_text': 'Is there a person in the image?', 'ground_truth_answer': 'yes', 'answer_format': 'yes_no', 'hallucination_type': 'object'}


In [8]:
!git clone https://github.com/junyangwang0410/AMBER.git

with open("AMBER/data/query/query_discriminative-attribute.json") as f:
    amber_attribute_q = json.load(f)
with open("AMBER/data/query/query_discriminative-relation.json") as f:
    amber_relation_q = json.load(f)
with open("AMBER/data/annotations.json") as f:
    amber_ann = json.load(f)

amber_ann_by_id = {a["id"]: a for a in amber_ann}


def build_amber_questions(query_list: list, hallucination_type: str) -> list:
    rows = []
    for q in query_list:
        ann = amber_ann_by_id[q["id"]]
        rows.append({
            "image_id": os.path.splitext(q["image"])[0],
            "question_id": f"amber_{q['id']}",
            "question_text": q["query"],
            "ground_truth_answer": ann["truth"],
            "answer_format": "yes_no",
            "hallucination_type": hallucination_type,
        })
    return rows


amber_questions = (
    build_amber_questions(amber_attribute_q, "attribute")
    + build_amber_questions(amber_relation_q, "relation")
)
print(len(amber_questions), "AMBER attribute+relation questions")
print(amber_questions[0])

fatal: destination path 'AMBER' already exists and is not an empty directory.
9292 AMBER attribute+relation questions
{'image_id': 'AMBER_1', 'question_id': 'amber_1005', 'question_text': 'Is the sky sunny in this image?', 'ground_truth_answer': 'yes', 'answer_format': 'yes_no', 'hallucination_type': 'attribute'}


In [9]:
AMBER_IMAGE_DIR = "/kaggle/input/datasets/nocturnalnerd18/amber-hallucination/image"

In [15]:
"""
!wget https://cs.stanford.edu/people/rak248/VG_100K/images.zip -O /kaggle/working/VG_100K.zip
!wget https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip -O /kaggle/working/VG_100K_2.zip
"""
!unzip -q -o /kaggle/working/VG_100K.zip -d /kaggle/temp/
!unzip -q -o /kaggle/working/VG_100K_2.zip -d /kaggle/temp/

In [11]:
!git clone https://github.com/JackChen-seu/Reefknot.git

def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

reefknot_yesno = load_jsonl("Reefknot/Dataset/YESNO.jsonl")
reefknot_mcq = load_jsonl("Reefknot/Dataset/Multichoice.jsonl")
reefknot_all = reefknot_yesno + reefknot_mcq

reefknot_by_image = defaultdict(list)
for r in reefknot_all:
    reefknot_by_image[r["image_id"]].append(r)

print(len(reefknot_by_image), "unique images across YESNO+MCQ,", len(reefknot_all), "questions total")

fatal: destination path 'Reefknot' already exists and is not an empty directory.
8827 unique images across YESNO+MCQ, 16690 questions total


In [12]:
def stratified_image_sample(by_image: dict, target_total: int, seed: int = 42) -> list:
    """Sample images (not questions) preserving the perception:cognitive ratio, using
    each image's majority relation_type as its bucket."""
    image_majority = {}
    for img_id, qs in by_image.items():
        types = Counter(q["relation_type"] for q in qs)
        image_majority[img_id] = types.most_common(1)[0][0]

    perception_imgs = [i for i, t in image_majority.items() if t == "perception"]
    cognitive_imgs = [i for i, t in image_majority.items() if t == "cognitive"]

    perception_frac = len(perception_imgs) / len(by_image)
    n_perception = round(target_total * perception_frac)
    n_cognitive = target_total - n_perception

    random.seed(seed)
    return random.sample(perception_imgs, n_perception) + random.sample(cognitive_imgs, n_cognitive)


REEFKNOT_TARGET_IMAGES = 500
reefknot_sampled_images = set(stratified_image_sample(reefknot_by_image, REEFKNOT_TARGET_IMAGES))
print(len(reefknot_sampled_images), "images sampled")
print(sum(len(reefknot_by_image[i]) for i in reefknot_sampled_images), "questions in the sample")

500 images sampled
938 questions in the sample


In [13]:
def build_reefknot_questions(by_image: dict, sampled_images: set) -> list:
    rows = []
    for img_id in sampled_images:
        for i, q in enumerate(by_image[img_id]):
            is_mcq = q["type"] == "Multichoice"
            rows.append({
                "image_id": img_id,
                "question_id": f"reefknot_{img_id}_{i}",
                "question_text": q["query_prompt"],
                "ground_truth_answer": q["label"],
                "answer_format": "mcq" if is_mcq else "yes_no",
                "hallucination_type": "relation",
                "relation_type": q["relation_type"],  # perception / cognitive -- kept for later breakdowns
            })
    return rows


reefknot_questions = build_reefknot_questions(reefknot_by_image, reefknot_sampled_images)
print(len(reefknot_questions), "Reefknot questions")
print(reefknot_questions[0])

938 Reefknot questions
{'image_id': '2336641', 'question_id': 'reefknot_2336641_0', 'question_text': 'What is the relation with man and man in this photo? A. lying B. next C. following D. past, please choose.', 'ground_truth_answer': 'B', 'answer_format': 'mcq', 'hallucination_type': 'relation', 'relation_type': 'perception'}


In [29]:
VG_IMAGE_DIRS = ["/kaggle/temp", "/kaggle/temp/VG_100K_2"]

def find_vg_image_path(image_id: str) -> str:
    for d in VG_IMAGE_DIRS:
        candidate = os.path.join(d, f"{image_id}.jpg")
        if os.path.exists(candidate):
            return candidate
    return None

In [30]:
def make_split_manifest(image_ids: list, train=0.7, val=0.15, seed: int = 42) -> dict:
    ids = list(image_ids)
    random.seed(seed)
    random.shuffle(ids)
    n = len(ids)
    n_train = int(n * train)
    n_val = int(n * val)
    manifest = {}
    for i, img_id in enumerate(ids):
        if i < n_train:
            manifest[img_id] = "train"
        elif i < n_train + n_val:
            manifest[img_id] = "val"
        else:
            manifest[img_id] = "test"
    return manifest


pope_image_ids = sorted({q["image_id"] for q in pope_questions})
amber_image_ids = sorted({q["image_id"] for q in amber_questions})
reefknot_image_ids = sorted(reefknot_sampled_images)

manifests = {
    "pope_split.json": make_split_manifest(pope_image_ids),
    "amber_split.json": make_split_manifest(amber_image_ids),
    "reefknot_split.json": make_split_manifest(reefknot_image_ids),
}

os.makedirs("/kaggle/working/splits", exist_ok=True)
for fname, manifest in manifests.items():
    out_path = os.path.join("/kaggle/working/splits", fname)
    with open(out_path, "w") as f:
        json.dump(manifest, f)
    counts = Counter(manifest.values())
    print(fname, dict(counts))

pope_split.json {'train': 350, 'val': 75, 'test': 75}
amber_split.json {'train': 702, 'val': 150, 'test': 152}
reefknot_split.json {'train': 350, 'val': 75, 'test': 75}


In [31]:
os.makedirs("/kaggle/working/questions", exist_ok=True)

all_questions = pope_questions + amber_questions + reefknot_questions

with open("/kaggle/working/questions/pope_questions.json", "w") as f:
    json.dump(pope_questions, f)
with open("/kaggle/working/questions/amber_questions.json", "w") as f:
    json.dump(amber_questions, f)
with open("/kaggle/working/questions/reefknot_questions.json", "w") as f:
    json.dump(reefknot_questions, f)
with open("/kaggle/working/questions/all_questions.json", "w") as f:
    json.dump(all_questions, f)

print(len(all_questions), "total discriminative questions across all three datasets:")
print(Counter(q["hallucination_type"] for q in all_questions))

!kaggle datasets init -p /kaggle/working/questions/
!kaggle datasets init -p /kaggle/working/splits/

13230 total discriminative questions across all three datasets:
Counter({'attribute': 7628, 'object': 3000, 'relation': 2602})
Data package template written to: /kaggle/working/questions/dataset-metadata.json
Data package template written to: /kaggle/working/splits/dataset-metadata.json


In [33]:
!kaggle datasets create -p /kaggle/working/questions/
!kaggle datasets create -p /kaggle/working/splits/

Starting upload for file reefknot_questions.json
100%|████████████████████████████████████████| 264k/264k [00:00<00:00, 1.15MB/s]
Upload successful: reefknot_questions.json (264KB)
Starting upload for file all_questions.json
100%|██████████████████████████████████████| 2.69M/2.69M [00:00<00:00, 13.7MB/s]
Upload successful: all_questions.json (3MB)
Starting upload for file pope_questions.json
100%|████████████████████████████████████████| 621k/621k [00:00<00:00, 2.71MB/s]
Upload successful: pope_questions.json (621KB)
Starting upload for file amber_questions.json
100%|██████████████████████████████████████| 1.82M/1.82M [00:00<00:00, 8.16MB/s]
Upload successful: amber_questions.json (2MB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/nocturnalnerd18/vlm-questions
Starting upload for file pope_split.json
100%|██████████████████████████████████████| 18.3k/18.3k [00:00<00:00, 95.0kB/s]
Upload successful: pope_split.json (18KB)
Starting uploa